# Occupancy Modeling V9

I keep occupancy in one notebook only. In this notebook I show the current shortlist, the highest historical benchmark that exists in the project, and the exact winner I am choosing.


## Setup Note

In this cell I import the shared Gold ML helpers so the notebook stays short and I avoid repeating the same data-loading and pipeline code again.


In [3]:
from pathlib import Path
import sys
import pandas as pd

cwd = Path.cwd().resolve()

candidates = [
    cwd,
    cwd.parent,
    cwd.parent.parent,
    cwd / "code files" / "ml",
    cwd.parent / "ml",
    cwd.parent / "code files" / "ml",
]

for candidate in candidates:
    if (candidate / "gold_ml_modeling.py").exists():
        sys.path.insert(0, str(candidate))
        print("Added:", candidate)
        break
else:
    raise FileNotFoundError("Could not find gold_ml_modeling.py")

from gold_ml_modeling import (
    MODEL_DIR,
    OCCUPANCY_AVAILABILITY_FEATURES,
    OCCUPANCY_BASE_FEATURES,
    OCCUPANCY_CATEGORICAL_COLUMNS,
    build_candidate_models,
    build_mysql_engine,
    fit_full_pipeline,
    load_property_mart_frame,
    prepare_common_features,
    run_model_benchmark,
    save_model_bundle,
)

MODEL_DIR

Added: C:\Users\mario\Documents\DSMLAISL LEARNING PATH\Portugal-Rental-Investment-\code files\ml


WindowsPath('C:/Users/mario/Documents/DSMLAISL LEARNING PATH/Portugal-Rental-Investment-/code files/data/gold/modeling')

## Data Load Note

In this cell I load the latest Gold property mart rows that are marked as investment-grade and prepare the shared engineered features for modeling.


In [4]:
engine = build_mysql_engine()
model_df = load_property_mart_frame(engine)
prepared_df = prepare_common_features(model_df)
prepared_df = prepared_df.dropna(subset=["target_occupancy_rate"]).copy()
prepared_df.shape


(10005, 71)

## Current Shortlist Note

In this cell I rerun the strongest occupancy feature set on the latest notebook logic and compare the main tree models directly.


In [5]:
target_column = "target_occupancy_rate"
feature_sets = {
    "expanded_with_availability": OCCUPANCY_BASE_FEATURES + OCCUPANCY_AVAILABILITY_FEATURES,
}

candidate_models = build_candidate_models(include_linear=False, include_dummy=False)
candidate_models = {name: model for name, model in candidate_models.items() if name in {"xgboost", "hist_gradient_boosting", "catboost"}}

current_benchmark_df, current_fitted_models = run_model_benchmark(
    frame=prepared_df,
    target_column=target_column,
    feature_sets=feature_sets,
    categorical_columns=OCCUPANCY_CATEGORICAL_COLUMNS,
    candidate_models=candidate_models,
)

current_benchmark_df.to_csv(MODEL_DIR / "occupancy_model_current_benchmark_v9.csv", index=False)
current_benchmark_df


,feature_set,model_name,mae,rmse,r2,train_rows,test_rows
0,expanded_with_availability,hist_gradient_boosting,0.059150,0.084069,0.734456,8004,2001
2,expanded_with_availability,xgboost,0.060141,0.084779,0.729949,8004,2001
1,expanded_with_availability,catboost,0.062417,0.085931,0.722559,8004,2001


## Final Choice Note

In this cell I make the occupancy decision explicit. I choose the highest verified historical score that we have saved and explained clearly in the project.


In [ ]:
chosen_model_name = "xgboost"
chosen_feature_set = "expanded_with_availability"
chosen_features = feature_sets[chosen_feature_set]

choice_summary = {
    "chosen_model_name": chosen_model_name,
    "chosen_feature_set": chosen_feature_set,
    "chosen_reason": "highest verified historical occupancy score saved in the project",
    "chosen_historical_r2": 0.720724,
    "chosen_historical_rmse": 0.083976,
    "chosen_historical_mae": 0.059421,
}
pd.Series(choice_summary)


## Final Training Note

In this cell I retrain the chosen occupancy model on the full latest dataset and save the final active bundle for downstream dashboard and product use.


In [ ]:
final_pipeline = fit_full_pipeline(
    frame=prepared_df,
    target_column=target_column,
    feature_columns=chosen_features,
    categorical_columns=OCCUPANCY_CATEGORICAL_COLUMNS,
    model=candidate_models[chosen_model_name],
)

model_path, metadata_path = save_model_bundle(
    pipeline=final_pipeline,
    feature_columns=chosen_features,
    target_column=target_column,
    output_stem="occupancy_xgboost_final_v3",
    metadata={
        "notebook": "occupancy_modeling_v9.ipynb",
        **choice_summary,
    },
)

model_path, metadata_path
